# JAX NumPy (`jax.numpy`) tutorial

An overview for basic scientific/engineering work.

**Goal**: mirror the operations in `numpy_basics.ipynb`, but using JAX.

**Core idea**: `jax.numpy` (`jnp`) looks like NumPy, but arrays are (conceptually) immutable, randomness uses explicit keys, and computation can be JIT-compiled / run on GPU/TPU.

**Companion (NumPy-only)**: `numpy_basics.ipynb` — same topics without JAX.

## Outline

1. Doc, scalars, vectorized `jnp` ops, trig
2. Arrays: creation, indexing, **immutable** updates (`.at`), combine
3. Element-wise ops, **linear algebra**, reductions, reshape, broadcasting
4. **PRNG** (`random.split`) and a training-loop sketch
5. **JAX transforms**: `jit`, `grad`, `value_and_grad`, Jacobians, `vmap`
6. **Advanced** (quick intros): PyTrees, `scan`, `jit` sharp bits, float64, timing, sharding, `jax.nn`, NumPy vs `jnp`, ecosystem


## Import

In Colab you may need `pip install jax jaxlib` (CPU) or a CUDA-specific install. Locally, use your usual env manager.


In [ ]:
import jax
import jax.numpy as jnp
from jax import random

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())


## Doc and Help

- JAX docs: `https://jax.readthedocs.io/`
- `jax.numpy` reference: `https://jax.readthedocs.io/en/latest/jax.numpy.html`

From code you can still use `help`.

**Same as NumPy**: `help(jnp.cos)` works the same.


In [ ]:
help(jnp.cos)


## Basic built-in math operations

**Same as NumPy**: Python scalars behave the same.


In [ ]:
u, v, w = 9, 2, -5
c1 = u + v
c2 = u - v
c3 = u * v
c4 = u / v
c5 = u ** v

print(c1, c2, c3, c4, c5)


## Basic math operations from `jax.numpy`

**Mostly the same as NumPy**: function names and broadcasting rules match.

**Different sometimes**:
- **Dtypes**: JAX defaults can differ (e.g. 32-bit vs 64-bit depending on config).
- **Arrays live on devices** (CPU/GPU/TPU) and operations may be staged/compiled.


In [ ]:
c6 = jnp.sqrt(u)
c7 = jnp.exp(v)
c8 = jnp.log(v)
c9 = jnp.log10(v)
c10 = jnp.log2(v)
c11 = abs(w)          # Python built-in, same
c12 = jnp.sign(w)
print(c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12)


## Tips for faster computation

**Same spirit as NumPy**: use array ops instead of Python loops.

**More JAX-specific**: you can additionally `jit` compile whole functions.


In [ ]:
inputs = jnp.array([1, 2, 3, 4, 5, 6])
outputs = jnp.exp(inputs)
print(outputs)


In [ ]:
@jax.jit
def f(x):
    return jnp.sin(x) + jnp.log(x + 0.1)

print(f(inputs))


## Trigo functions

**Same as NumPy**: `radians`, `degrees`, `cos`, `sin`, `tan`.


In [ ]:
# Trigonometry
ang_deg = 30.0
ang_rad = jnp.radians(ang_deg)
and_deg = jnp.degrees(ang_rad)
print('Deg: ', ang_deg)
print('Radian: ', ang_rad)

y1 = jnp.cos(ang_rad)
y3 = jnp.sin(ang_rad)
y5 = jnp.tan(ang_rad)

print(y1, y3, y5)


In [ ]:
# Inverse Trigonometric Functions
x1 = jnp.arccos(y1)
x3 = jnp.arcsin(y3)
x5 = jnp.arctan(y5)

print(x1, x3, x5)


In [ ]:
# Atan vs Atan2
ang_deg = -150

s = jnp.sin(jnp.radians(ang_deg))
c = jnp.cos(jnp.radians(ang_deg))
t = s / c

z1 = jnp.degrees(jnp.arctan(t))
z2 = jnp.degrees(jnp.arctan2(s, c))

print(f'z1 = {z1}')
print(f'z2 = {z2}')


Make sure that you understand the difference between `arctan` and `arctan2`.


## Arrays

Vectors, matrices, tensors (n-d arrays).

**Same as NumPy**: `jnp.array`, `ndim`, `shape`.

**Different**: JAX arrays are **immutable**. You can’t do `A[i, j] = ...`; use `.at[...]` updates.


In [ ]:
# Init from data
data = [[1, 2], [3, 4]]
A = jnp.array(data)
print(A)

print(f'A = \n {A}')
print(f'A is an array of dimension {A.ndim} and of shape {A.shape}')


In [ ]:
# 1D vs 2D vectors
x_1d = jnp.array([1, 2, 3])
x_2d_row = jnp.array([[1, 2, 3]])
x_2d_col = jnp.array([[1], [2], [3]])
print(f'\n x_1d = \n {x_1d}')
print(f'x_1d is an array of dimension {x_1d.ndim} and of shape {x_1d.shape}')
print(f'\n x_2d_row = \n {x_2d_row}')
print(f'x_2d_row is an array of dimension {x_2d_row.ndim} and of shape {x_2d_row.shape}')
print(f'\n x_2d_col = \n {x_2d_col}')
print(f'x_2d_col is an array of dimension {x_2d_col.ndim} and of shape {x_2d_col.shape}')


In [ ]:
# Standard init
eye = jnp.eye(4)

shape = (2, 3)
ones = jnp.ones(shape)
zeros = jnp.zeros(shape)

print(f"Identity  (diag): \n {eye} \n")
print(f"Ones : \n {ones} \n")
print(f"Zeros : \n {zeros} \n")

# Sequence init
x1 = jnp.arange(0, 10, 1)  # start, stop, step
x2 = jnp.linspace(0, 9, 10)  # start, end, numbers of elements
print(f"Arange: \n {x1} \n")
print(f"Linspace: \n {x2} \n")


In [ ]:
# Attributes (and JAX-specific: device)
key = random.PRNGKey(0)
x = random.normal(key, shape=(3, 4, 3))
print(f"Shape : {x.shape}")
print(f"Number of dimensions: {x.ndim}")
print(f"Datatype : {x.dtype}")
print(f"Device : {x.device}")


## Accessing values

**Same as NumPy**: indexing/slicing (`v[0]`, `v[-1]`, `v[1:4]`, `A[i, j]`, `A[:, 0]`, etc.).

The cell after the example expands **advanced indexing** (Python list indices vs `jnp.array`, and `jit`).


In [ ]:
# Accessing elements of a row vector
v = jnp.array([1, 3, 7, 8, 11, 14])
print(f'v         = {v}')
print(f'v[0]      = {v[0]}')  # first element
print(f'v[-1]     = {v[-1]}')  # last element
print(f'v[1:4]    = {v[1:4]}')  # start with 1, stop before 4

# sub-vector
print(f'v         = {v}')
# print(f'v[[1,3]] (list indices) = {v[[1, 3]]}')  # often works in eager mode
print(f'v[jnp.array([1,3])]      = {v[jnp.array([1, 3])]}')  # explicit array indices
print(f'v[1::2]   = {v[1::2]}')
print(f'v[3:]     = {v[3:]}')

# Accessing elements of a 2D matrix
A = jnp.array([[1, 2, 3, 4, 5],
               [6, 7, 8, 9, 10],
               [11, 12, 13, 14, 15],
               [16, 17, 18, 19, 20]])

i = 2
j = 3
print(f"Element at row i column j: {A[i, j]}")
print(f"First row: {A[0, :]}")
print(f"First column: {A[:, 0]}")
print(f"Last column: {A[:, -1]}")


### Advanced indexing (list vs `jnp.array`)

**Advanced indexing** = picking several elements at once with an index list/array (e.g. elements at positions 1 and 3).

- **Python list indices often work** in normal (eager) execution, e.g. `v[[1, 3]]` is usually fine.
- **Prefer `jnp.array([...])` as indices** when indices come from **JAX computations** or inside **`jax.jit`**: traced values should be arrays, not Python lists you build dynamically.
- **Rule of thumb**: fixed literal indices → list-style is OK; indices produced by JAX (especially under `jit`) → use `idx = jnp.array([...])` (or `jnp.where`, `argmax`, etc.) and then `v[idx]`.


## Attribution of elements (mutation)

**Different from NumPy**: JAX arrays are immutable, so direct assignment like `A[2,2] = 100` fails.

Use functional updates with `.at[...]`.

**Out-of-place**: each update returns a **new** array; the original `A` is unchanged (unlike in-place `A[i,j] = ...` in NumPy).


In [ ]:
A = jnp.array([[1, 2, 3, 4, 5],
               [6, 7, 8, 9, 10],
               [11, 12, 13, 14, 15],
               [16, 17, 18, 19, 20]])

print(f"Original matrix: \n {A} \n")

A2 = A.at[2, 2].set(100)
A2 = A2.at[:, 1].set(-1)   # whole column
A2 = A2.at[-1, :].set(0)   # whole row

print(f"Modified matrix (new array): \n {A2} \n")
print(f"Original unchanged: \n {A} \n")


## Combining arrays

**Same as NumPy**: `concatenate`, `stack`.

**Different-ish**: `jnp.append` exists, but is just a convenience wrapper; prefer `concatenate`/`stack` for clarity.


### `append`

`jnp.append` exists (like NumPy). **Out-of-place** means: it does **not** resize or modify `x` in memory; it **returns a new array** with the extra row/column. (Same idea as `.at[...].set(...)`: originals stay unchanged.)

For clarity and performance, **`jnp.concatenate`** / **`jnp.stack`** are usually preferred over `append`.


In [ ]:
x = jnp.array([[1, 2], [3, 4]])
u = jnp.array([[7, 7]])

X1 = jnp.append(x, u.T, axis=1)  # add column
print(f'With an added column: \n{X1}')
X2 = jnp.append(x, u, axis=0)    # add row
print(f'With an added row: \n{X2}')


In [ ]:
data = [[1, 2], [3, 4]]
x = jnp.array(data)

print(f'x =\n {x} \n with shape {x.shape}')

# Existing dimensions (cat)
print('Concatenate along axis 0')
data0 = jnp.concatenate([x, x, x], axis=0)
print(f'[x; x; x] =\n {data0} \n with shape {data0.shape}')

print('Concatenate along axis 1')
data1 = jnp.concatenate([x, x, x], axis=1)
print(f'[x x x] =\n {data1} \n with shape {data1.shape}')

# New dimension (stack)
data = jnp.stack([x, x, x], axis=0)
print(f'\n New array = \n{data} \n with shape {data.shape}')
data = jnp.stack([x, x, x], axis=1)
print(f'\n New array = \n{data} \n with shape {data.shape}')
data = jnp.stack([x, x, x], axis=2)
print(f'\n New array = \n{data} \n with shape {data.shape}')

# Append-like examples (use concatenate)
u = jnp.array([[7, 7]])
X3 = jnp.concatenate([x, u.T], axis=1)
print(f'With an added column: \n{X3}')

X4 = jnp.concatenate([x, u], axis=0)
print(f'With an added row: \n{X4}')


## Array operations

Warning (same as NumPy): `*` is element-wise, not matrix multiplication.


In [ ]:
data = [[1, 2, 3]]
x = jnp.array(data).T
A = jnp.eye(3) * 2

print(x)
print(A)

# Element-wise basic operation
A2 = A + A
print(A2)

x2 = x + 2 * x - 1.0 * x ** 2
print(x2)

# Element-wise numpy operation (same names)
A2 = jnp.sin(A) + jnp.sign(A) + jnp.log(A + 0.1)
print(A2)

x2 = jnp.exp(x)
print(x2)


## Linear algebra

**Same as NumPy**: treat vectors as 2D arrays for “matrix math”; use `@` for matrix multiply.

**Mostly same API**: `jnp.linalg.inv/pinv/solve/norm/lstsq`.


In [ ]:
data = [[1, 2, 3]]  # double bracket => 2D array
x = jnp.array(data).T  # column vector
A = jnp.eye(3) * 2
A = A.at[1, 2].set(1)

print(f'Matrix A = \n{A}')
print(f'Column vector x = \n{x}')

print(f'Number of dimension of A = {A.ndim}')
print(f'Number of dimension of x = {x.ndim}')

print(f'Shape of A = {A.shape}')
print(f'Shape of x = {x.shape}')

# Matrix multiplication
y = A @ x
print(f'y = A @ x = \n{y}')

# Scalar product (inner)
xTx = x.T @ x
print(xTx)

print(x.T * x)  # element-wise, not inner product

print(jnp.linalg.norm(x) ** 2)

# Outer/Dyadic product
xxT = x @ x.T
print(xxT)

# Matrix inversion
A_inv = jnp.linalg.inv(A)
print(A_inv)
print(A @ A_inv)

# Pseudo-inverse
A_pinv = jnp.linalg.pinv(A)
print(A_pinv)

# Solve linear system
b = jnp.array([[1, 2, 3]]).T
x_sol = A_inv @ b
print(x_sol)
print(A @ x_sol)

# Using solve for a linear system of equation
x_sol2 = jnp.linalg.solve(A, b)
print(x_sol2)

# Least-square solution
# Note: JAX returns a tuple; solution is the first element (same idea as NumPy indexing [0])
x_ls = jnp.linalg.lstsq(A, b)[0]
print(x_ls)


## Sum / Min / Max

**Same as NumPy**: `sum`, `min`, `max`, `argmin`, `argmax`, and `axis=`.


In [ ]:
data = [[1, 2, 10], [3, 4, 0], [3, 1, 1]]
x = jnp.array(data)

print(x)
print("Summation per colum: ", jnp.sum(x, axis=0))
print("Summation per row: ", jnp.sum(x, axis=1))

print("Min per colum: ", x.min(axis=0))
print("Min per row: ", x.min(axis=1))

print("ArgMin per colum: ", x.argmin(axis=0))
print("ArgMin per row: ", x.argmin(axis=1))

print("Max per colum: ", x.max(axis=0))
print("Max per row: ", x.max(axis=1))

print("ArgMax per colum: ", x.argmax(axis=0))
print("ArgMax per row: ", x.argmax(axis=1))

print("Row index that maximize the sum over column: ", x.sum(axis=0).argmax())


## Reshaping

**Same as NumPy**: `flatten`, `reshape`, and `-1` for automatic dimension.


In [ ]:
tensor = jnp.array([[[1, 2, 3], [4, 5, 6]], [[2, 2, 2], [3, 3, 3]]])
print(f"Tensor (3D-array):\n {tensor} with shape {tensor.shape}\n")

x = tensor.flatten()
print(f"Vector (1D-array) with the same information:\n {x} with shape {x.shape}\n")

matrix = tensor.reshape(2, -1)  # -1 means automatic
print(f"Matrix (2D-array) with the same information:\n {matrix} with shape {matrix.shape}\n")

tensor2 = matrix.reshape(2, 2, -1)
print(f"Reconstructed tensor:\n {tensor2} with shape {tensor2.shape}\n")


## Broadcasting for parallel operations

**Same as NumPy**: broadcasting rules and use of `None`/`newaxis`.


In [ ]:
x = jnp.array([1, 2, 3])
y = jnp.array([0.1, 0.2, 0.1, 0.4, 0.5])

z = x[:, None] * y[None, :]

print(z)  # z_ij = x_i * y_j for all (i, j)


## Random number generation

**Different from NumPy**: JAX uses **explicit PRNG keys** (pure functional RNG). To get new random numbers, you must **split** the key.

NumPy style:
- `np.random.uniform(size=shape)`
- `np.random.normal(size=shape)`

JAX style:
- `key = random.PRNGKey(seed)`
- `key, sub = random.split(key)`
- `random.uniform(sub, shape=...)`


In [ ]:
shape = (2, 3)

key = random.PRNGKey(0)
key, k1 = random.split(key)
key, k2 = random.split(key)

rand_tensor = random.uniform(k1, shape=shape)
randn_tensor = random.normal(k2, shape=shape)

print(f"Random Tensor (uniform): \n {rand_tensor} \n")
print(f"Random Tensor (normal): \n {randn_tensor} \n")


### PRNG in training loops

Unlike NumPy’s global RNG, **carry the key forward**: each step, **`split`** to get a fresh subkey for randomness and keep the updated `key` for the next iteration. That keeps draws **reproducible** and avoids reusing the same subkey twice.


In [ ]:
# Sketch: one training step consumes a new subkey for noise/data augmentation
def train_step(key, params):
    key, k_noise = random.split(key)
    noise = random.normal(k_noise, (4,))
    # ... use noise, update params ...
    return key, params, noise

key = random.PRNGKey(0)
params = ()
for _ in range(3):
    key, params, _ = train_step(key, params)


## JAX special features

The sections above focus on `jax.numpy` parity with NumPy. This section covers features that are central to JAX itself: compilation, autodiff, and vectorization.


### Compilation with `jax.jit`

`jax.jit` traces your function and compiles it (XLA). After first-call compilation, repeated calls are often much faster.

**Rule of thumb**:
- Put numeric work in pure functions.
- Avoid Python side effects inside jitted functions.


In [ ]:
def poly(x):
    return jnp.sin(x) + 0.1 * x**2 - jnp.log(x + 1.0)

poly_jit = jax.jit(poly)

x = jnp.linspace(0.1, 10.0, 8)
print("Without jit:", poly(x))
print("With jit   :", poly_jit(x))


### Automatic differentiation: `jax.grad`

Unlike NumPy, gradients are first-class in JAX.

`jax.grad(f)` returns a new function that computes `df/dx` for a scalar-output `f`.


In [ ]:
def f(x):
    # scalar-valued function of one variable: R -> R
    return x**2 + 3.0 * x + jnp.sin(x)

grad_f = jax.grad(f)

a = 2.0
print("f(a) =", f(a))
print("f'(a) =", grad_f(a))


### `value_and_grad`

Returns both the value `f(a)` and the derivative `f'(a)` in one call.


In [ ]:
value_and_grad_f = jax.value_and_grad(f)
val, g = value_and_grad_f(a)
print("f(a) =", val)
print("f'(a) =", g)


### Jacobian of vector-valued functions: `jax.jacfwd` / `jax.jacrev`

For `f: R^n → R^m`, the Jacobian is \(J_{ij} = \partial f_i / \partial x_j\).

| | **`jax.jacfwd`** | **`jax.jacrev`** |
|---|------------------|------------------|
| **Mode** | Forward-mode AD | Reverse-mode AD |
| **Often cheaper when** | **Few inputs**, many outputs | **Few outputs**, many inputs (e.g. scalar `f` → many variables) |

For a **square** map (e.g. `R² → R²`), both return the same matrix up to normal float noise; for large problems, pick the mode that matches the cheap direction.


In [ ]:
def f_vec(x):
    # x shape (2,) -> output shape (2,)
    return jnp.array([
        x[0] * x[1],
        x[0] ** 2 + jnp.sin(x[1])
    ])

xv = jnp.array([2.0, 0.5])
J_fwd = jax.jacfwd(f_vec)(xv)
J_rev = jax.jacrev(f_vec)(xv)

print("Jacobian (jacfwd):\n", J_fwd)
print("Jacobian (jacrev):\n", J_rev)


### Automatic batching with `jax.vmap`

`vmap` vectorizes a function without writing explicit Python loops.

**Batch vs per-sample shape** (default `in_axes=0`):

- The **first axis** of each batched input is the **batch dimension** (which sample).
- The **remaining axes** are what the inner function sees for **one** sample (e.g. a vector of length 3).

Example: `batch_x` with shape `(3, 3)` → batch size 3, each `x` has shape `(3,)`. If your data is stored with batch on another axis, use `in_axes=...` or transpose so the batch axis is the one you map over.


In [ ]:
def single_sample_fn(x):
    # x shape (3,)
    return jnp.sum(jnp.tanh(x) ** 2)

batch_x = jnp.array([
    [0.1, 0.2, 0.3],
    [1.0, 2.0, 3.0],
    [-0.5, 0.7, 1.2],
])

batched_fn = jax.vmap(single_sample_fn)
print("Per-sample outputs:", batched_fn(batch_x))


### Compose transformations: `jit` and `grad`

JAX transformations compose in either order; both are valid.

| Order | Meaning |
|--------|--------|
| **`jax.jit(jax.grad(f))`** | Compile the **derivative** map `f'` (one compiled backward pass). |
| **`jax.grad(jax.jit(f))`** | **Differentiate** a function whose **forward** is `jit`’d. |

For a scalar `f`, you often use **`jit`** around **`value_and_grad`** so `f` and `f'` run in one compiled step.


In [ ]:
jit_grad_f = jax.jit(jax.grad(f))
print("jit(grad(f))(a) =", jit_grad_f(a))

grad_jit_f = jax.grad(jax.jit(f))
print("grad(jit(f))(a) =", grad_jit_f(a))


### Bonus: control flow with `jax.lax`

For loops/branches that should be compiled robustly under `jit`, JAX provides primitives like `jax.lax.fori_loop`, `jax.lax.scan`, `jax.lax.cond`.

For many simple cases, regular Python control flow works, but `lax` is the most explicit/portable option for staged computation.


## Advanced topics (quick intros)

Short pointers to ideas you will see in larger JAX codebases. Each subsection is only a **taste**; follow the official docs for details.

**In this section**: PyTrees → `lax.scan` → `jit` static args & `debug.print` → float64 → `block_until_ready` → multi-device → `jax.nn` → NumPy vs `jnp` → ecosystem packages.


### PyTrees and `jax.tree_util`

JAX treats **nested Python structures** (dicts, lists, tuples of arrays) as **PyTrees**. Gradients, optimizers, and many libraries (`optax`, `equinox`, `flax`) expect parameters in this form—not always one flat vector.

Useful helpers: `jax.tree_util.tree_leaves`, `tree_map`, `tree_reduce`.


In [ ]:
import jax.tree_util as tu

params = {"w": jnp.array([1.0, 2.0]), "b": jnp.array(0.5)}

def scalar_energy(p):
    return jnp.sum(p["w"] ** 2) + p["b"] ** 2

g = jax.grad(scalar_energy)(params)
print("grad w.r.t. nested params:", g)
print("leaves:", tu.tree_leaves(g))


### `jax.lax.scan` (carried loops)

`scan` implements a loop with a **carry** and a **sequence**: think integrators, RNN-style updates, or “fold over time” without a Python `for` inside `jit`.

Signature idea: `scan(f, init, xs)` where `f(carry, x) -> (new_carry, y)`.


In [ ]:
def step(carry, x):
    new_carry = carry + x
    return new_carry, new_carry  # (carry_out, stacked_output)

xs = jnp.arange(1, 5, dtype=jnp.float32)
final_carry, ys = jax.lax.scan(step, 0.0, xs)
print("final sum:", final_carry)
print("partial sums:", ys)


### `jit` details: static arguments and debugging

- **`static_argnums` / `static_argnames`**: tell `jit` which arguments are **Python values** (not traced arrays), e.g. an integer loop count or feature size. Those can change which compiled version runs.
- **`jax.debug.print`**: preferred way to print **inside** `jit` (plain `print` is easy to misuse with tracing).


In [ ]:
from functools import partial

@partial(jax.jit, static_argnames=("n",))
def monomial(x, n):
    return x ** n

print("x**3:", monomial(jnp.array(2.0), 3))

@jax.jit
def with_debug(x):
    jax.debug.print("x={}", x)
    return x + 1.0

_ = with_debug(jnp.array(1.0))


### Precision: float64 (`x64`)

JAX often defaults to **float32**. For NumPy-like **float64**, enable at **process start**:

```python
jax.config.update("jax_enable_x64", True)
```

Then construct arrays as `jnp.array(..., dtype=jnp.float64)` where needed.

Prefer enabling **once at the start** of a session (before large work); changing mid-session can be confusing.


In [ ]:
# Default floating dtype in many JAX setups is float32:
print("default:", jnp.array([1.0]).dtype)

# To use float64 globally, uncomment the next line in a **fresh** session and run it before other JAX work:
# jax.config.update("jax_enable_x64", True)
# print("after x64:", jnp.array([1.0]).dtype)


### Timing: `jax.block_until_ready`

Computation is **asynchronous**; without blocking, wall-clock time can look “too fast.” For benchmarks, call **`jax.block_until_ready(x)`** on the result (or use built-in benchmarking helpers in the docs).


In [ ]:
x = jnp.ones((2_000_000,))
y = jax.jit(lambda z: z + 1.0)(x)
y = jax.block_until_ready(y)
print("ready, shape:", y.shape)


### `jax.nn` (activations and stable ops)

Common building blocks for models: **`relu`**, **`swish`**, **`softmax`**, **`logsumexp`**, **`sigmoid`**, etc. They compose cleanly with **`grad`**.


In [ ]:
from jax.nn import relu, softmax

logits = jnp.array([1.0, 2.0, 3.0])
print("softmax:", softmax(logits))
print("relu:", relu(jnp.array([-1.0, 2.0])))


### NumPy vs `jax.numpy` in one pipeline

Prefer **`jnp`** inside functions you **`jit`** / **`grad`**. Plain **`numpy`** returns host arrays and breaks tracing if mixed carelessly. Rule of thumb: **`device_put`** / **`jnp.asarray`** to convert explicitly when crossing boundaries.


In [ ]:
import numpy as np

# Data from "NumPy land" (host CPU, numpy.ndarray)
a_np = np.array([1.0, 2.0, 3.0])

# Explicit conversion to JAX arrays before jit/grad
a_jax = jnp.asarray(a_np)
# or: a_jax = jax.device_put(a_np)

@jax.jit
def g(x):
    return jnp.sum(x ** 2)

print("g(jnp array):", g(a_jax))

# Bring a result back to NumPy if you need it outside JAX (e.g. plotting on CPU)
out = np.asarray(g(a_jax))
print("as numpy:", out, type(out))
